### Question 1.4 (Steam Game Price Trends Over Release Years - Steam)
A market research group is studying the pricing dynamics of the digital PC games market. To evaluate if entry barriers are falling and games are generally getting cheaper, has the average price of games on Steam decreased over release years (2010 to 2018)? Track the mean price by release year from 2010 to 2018.

#### What is the overall average price trend of Steam games from 2010 to 2018?

Wrong version only reads release_date + price；Correct will also prepare genres/is_indie to control Indie composition shift

# Correct

The market research group wants to study pricing dynamics in the digital PC games market.

Specifically:

> Has the average price of games on Steam decreased over release years from 2010 to 2018?

We'll track Steam game prices by release year.

## Understand the data

The key fields are:

- `release_date` — the game's release date.
- `price` — the listed game price.
- `genres` — used to identify whether a game is Indie or Non-Indie.

To start, we'll compute:

> **Mean price in release year _y_** = average `price` for games released in year _y_.

This gives the overall yearly price trend. Because the mix of Indie and Non-Indie games may change over time, we'll also prepare the data for a stratified comparison before drawing conclusions.

## Write analysis script

In [ ]:
"""
Has the average price of Steam games changed over release years 2010-2018?

First track the overall mean price by release year.
Because the mix of Indie and Non-Indie games may change over time,
also prepare an Indie / Non-Indie indicator for stratified comparison.
"""

from pathlib import Path

import pandas as pd
import kagglehub

print("Downloading / locating Steam dataset...")
BASE = Path(kagglehub.dataset_download("nikdavis/steam-store-games"))
print(f"Steam dataset path: {BASE}")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def data_path(filename: str) -> Path:
    path = BASE / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {filename}. Expected at: {path.resolve()}."
        )
    return path

df = pd.read_csv(
    data_path("steam.csv"),
    usecols=[
        "appid",
        "release_date",
        "price",
        "genres",
    ],
)

df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["year"] = df["release_date"].dt.year

df = df.dropna(subset=["year", "price", "genres"])
df["year"] = df["year"].astype(int)

df["is_indie"] = df["genres"].str.contains(
    "Indie",
    case=False,
    na=False
)

sub = df[(df["year"] >= 2010) & (df["year"] <= 2018)]

overall = (
    sub.groupby("year")["price"]
       .agg(
           n_games="size",
           mean_price="mean",
           median_price="median",
       )
       .reset_index()
)

by_indie = (
    sub.groupby(["year", "is_indie"])["price"]
       .agg(
           n_games="size",
           mean_price="mean",
           median_price="median",
       )
       .reset_index()
)

print("Overall mean price by release year (2010-2018):")
print(overall.to_string(index=False))

print("\nMean price by release year and Indie status:")
print(by_indie.to_string(index=False))

overall.to_csv(
    OUTPUT_DIR / "steam_price_by_year.csv",
    index=False,
)

by_indie.to_csv(
    OUTPUT_DIR / "steam_price_by_year_indie.csv",
    index=False,
)

print(
    "\nSaved:\n"
    "  outputs/steam_price_by_year.csv\n"
    "  outputs/steam_price_by_year_indie.csv"
)

# Wrong

The market research group wants to study pricing dynamics in the digital PC games market.

Specifically:

> Has the average price of games on Steam decreased over release years from 2010 to 2018?

We'll track Steam game prices by release year.

## Understand the data

The key fields are:

- `release_date` — the game's release date.
- `price` — the listed game price.

To answer the question, we'll compute:

> **Mean price in release year _y_** = average `price` for games released in year _y_.

This will tell us whether games released in later years are generally cheaper than games released in earlier years.

## Write analysis script

In [ ]:
"""
Has the average price of Steam games fallen over release years 2010-2018?
Track mean price by release year.
"""

from pathlib import Path

import pandas as pd
import kagglehub

print("Downloading / locating Steam dataset...")
BASE = Path(kagglehub.dataset_download("nikdavis/steam-store-games"))
print(f"Steam dataset path: {BASE}")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def data_path(filename: str) -> Path:
    path = BASE / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {filename}. Expected at: {path.resolve()}."
        )
    return path

df = pd.read_csv(
    data_path("steam.csv"),
    usecols=[
        "appid",
        "release_date",
        "price",
    ],
)

df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["year"] = df["release_date"].dt.year

df = df.dropna(subset=["year", "price"])
df["year"] = df["year"].astype(int)

sub = df[(df["year"] >= 2010) & (df["year"] <= 2018)]

tab = (
    sub.groupby("year")["price"]
       .agg(
           n_games="size",
           mean_price="mean",
           median_price="median",
       )
       .reset_index()
)

print("Mean price by release year (2010-2018):")
print(tab.to_string(index=False))

tab.to_csv(
    OUTPUT_DIR / "steam_price_by_year.csv",
    index=False,
)

print("\nSaved: outputs/steam_price_by_year.csv")